In [51]:
import torch
import torch.nn as nn

from torchvision import datasets, transforms
from torch.utils.data import DataLoader,TensorDataset

SEED = 42
EPSILON = 0.25

torch.manual_seed(SEED)

In [52]:
transform = transforms.ToTensor()

train = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform = transform
)

test = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform = transform
)


train_loader = DataLoader(
    train,
    batch_size=128,
    shuffle = True
)

test_loader = DataLoader(
    test,
    batch_size=128,
    shuffle=False
)

In [53]:
class Maxout(nn.Module):

  def __init__(self, in_features, out_features, pool_size):

    super().__init__()

    self.linear = nn.Linear(in_features, out_features*pool_size)

    self.in_feartures = in_features

    self.out_features = out_features

    self.pool_size = pool_size

  def forward(self,x):

    x = self.linear(x)

    x = x.view(x.size(0), self.out_features, self.pool_size)

    return x.max(dim=2).values

In [54]:
class MaxoutNet(nn.Module):

  def __init__(self,pool_size):
    super().__init__()

    self.maxout1 = Maxout(in_features=784,out_features=240,pool_size=pool_size)
    self.maxout2 = Maxout(in_features=240,out_features=240,pool_size=pool_size)
    self.output = nn.Linear(240,10)

    self.dropout_input = nn.Dropout(0.2)
    self.dropout1 = nn.Dropout(0.5)
    self.dropout2 = nn.Dropout(0.5)

  def forward(self,x):

    x = x.view(x.size(0), -1)
    x = self.dropout_input(x)
    x = self.maxout1(x)
    x = self.dropout1(x)
    x = self.maxout2(x)
    x = self.dropout2(x)
    x = self.output(x)

    return x


In [55]:
torch.manual_seed(SEED)
model = MaxoutNet(pool_size=5)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr = 0.001
)

epochs = 5

for epoch in range(epochs):

  model.train()
  total_loss = 0

  for x,y in train_loader:

    optimizer.zero_grad()

    output = model(x)

    loss = criterion(output,y)

    loss.backward()

    optimizer.step()

    total_loss += loss.item()

  avg_loss = total_loss/len(train_loader)

  print(f"Epoch {epoch + 1}, loss = {avg_loss:.4f}")


Epoch 1, loss = 0.4052
Epoch 2, loss = 0.1711
Epoch 3, loss = 0.1417
Epoch 4, loss = 0.1306
Epoch 5, loss = 0.1159


In [56]:
model.eval()

with torch.no_grad():

  total = 0
  correct = 0

  for x,y in test_loader:

    output = model(x)
    _,predicted = torch.max(output,dim=1)
    total += y.size(0)
    correct += (predicted == y).sum().item()

accuracy = correct / total

print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.9787


In [57]:
torch.save(model.state_dict(), "maxout_mnist.pth")

***Generating Adversary Examples & Accuracy Test***

In [58]:
epsilon = EPSILON
adv_x = []
adv_y = []

model.eval()

for x, y in test:

    x = x.unsqueeze(0)
    y = torch.tensor([y])
    x.requires_grad_()
    output = model(x)
    loss = criterion(output, y)
    model.zero_grad()
    loss.backward()

    nabla = x.grad
    attack = epsilon * nabla.sign()
    x_adv = x + attack
    x_adv = torch.clamp(x_adv,0,1)
    adv_x.append(x_adv.detach().squeeze(0))
    adv_y.append(y.squeeze(0))

adv_x = torch.stack(adv_x)

adv_y = torch.stack(adv_y)

adv_test = TensorDataset(adv_x, adv_y)

adv_test_loader = DataLoader(

    adv_test,

    batch_size=128,

    shuffle=False

)

total = 0
correct = 0

with torch.no_grad():

  for x,y in adv_test_loader:

    output = model(x)
    _,predicted = torch.max(output,dim=1)
    total += y.size(0)
    correct += (predicted == y).sum().item()

accuracy = correct / total

print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.0406


***Adversary Training***

In [59]:
def fgsm_attack(model, criterion, x, y, epsilon):

    x = x.clone().detach()
    x.requires_grad_()

    output = model(x)
    loss = criterion(output, y)

    model.zero_grad()
    loss.backward()

    attack = epsilon * x.grad.sign()

    x_adv = x + attack
    x_adv= torch.clamp(x_adv,0,1)
    return x_adv.detach()

In [60]:
def pgd_attack(model, criterion, s, y , epsilon, alpha, step):

  s_origin = s.clone().detach()
  noise = torch.empty_like(s_origin).uniform_(-epsilon, epsilon)
  s = torch.clamp(s_origin + noise, 0, 1).detach()

  for i in range(step):

    s.requires_grad_()
    output = model(s)
    loss = criterion(output, y)

    model.zero_grad()
    loss.backward()
    attack = alpha * s.grad.sign()
    s_adv = s + attack
    delta = s_adv - s_origin
    delta = torch.clamp(delta, -epsilon, epsilon)
    s_adv = torch.clamp(s_origin + delta, 0, 1)
    s = s_adv.detach()

  return s

In [61]:
torch.manual_seed(SEED)
model = MaxoutNet(pool_size=5)
epsilon = EPSILON
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001)

epochs = 5

for epoch in range(epochs):

  model.train()
  total_loss = 0

  for x,y in train_loader:

    x_adv = fgsm_attack(model, criterion, x, y, epsilon)

    optimizer.zero_grad()

    clean_loss = criterion(model(x),y)
    adv_loss = criterion(model(x_adv),y)
    loss = 0.5 * clean_loss + 0.5 * adv_loss

    loss.backward()
    optimizer.step()

    total_loss += loss.item()

  avg_loss = total_loss/len(train_loader)

  print(f"Epoch {epoch + 1}, loss = {avg_loss:.4f}")

  torch.save(model.state_dict(), "maxout_mnist_fgsm_adv_eps025.pth")


Epoch 1, loss = 0.8632
Epoch 2, loss = 0.4721
Epoch 3, loss = 0.3980
Epoch 4, loss = 0.3592
Epoch 5, loss = 0.3281


***Testing New Model***

In [62]:
model.eval()

adv_test_x = []
adv_test_y = []

for x, y in test:

  x = x.unsqueeze(0)
  y = torch.tensor([y])

  x_adv = fgsm_attack(
      model,
      criterion,
      x,
      y,
      epsilon
  )

  adv_test_x.append(x_adv.squeeze(0))
  adv_test_y.append(y.squeeze(0))

adv_test_x = torch.stack(adv_test_x)
adv_test_y = torch.stack(adv_test_y)

adv_test = TensorDataset(
    adv_test_x,
    adv_test_y
)

adv_test_loader = DataLoader(
    adv_test,
    batch_size=128,
    shuffle=False
)

In [63]:
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for x, y in adv_test_loader:
        output = model(x)
        predicted = output.argmax(dim=1)

        total += y.size(0)
        correct += (predicted == y).sum().item()

accuracy = correct / total

print(f"FGSM对抗训练后的FGSM准确率：{accuracy:.4f}")

FGSM对抗训练后的FGSM准确率：0.7281


In [64]:
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for x, y in test_loader:
        output = model(x)
        predicted = output.argmax(dim=1)

        total += y.size(0)
        correct += (predicted == y).sum().item()

print(f"FGSM对抗训练的干净准确率：{correct / total:.4f}")

FGSM对抗训练的干净准确率：0.9733


In [65]:
model.eval()

total = 0
correct = 0

for x, y in test_loader:

    x_adv = pgd_attack(
        model,
        criterion,
        x,
        y,
        epsilon=EPSILON,
        alpha=epsilon / 4,
        step=20
    )

    with torch.no_grad():
        output = model(x_adv)
        predicted = output.argmax(dim=1)

        total += y.size(0)
        correct += (predicted == y).sum().item()

accuracy = correct / total

print(f"FGSM 对抗训练模型的 PGD-20 准确率：{accuracy:.4f}")

FGSM 对抗训练模型的 PGD-20 准确率：0.0212


In [66]:
clean_model = MaxoutNet(pool_size=5)
clean_model.load_state_dict(
    torch.load("maxout_mnist.pth", map_location="cpu", weights_only=True)
)
clean_model.eval()
total = 0
correct = 0
for x,y in test_loader:

  x_adv = pgd_attack(
      clean_model,
      criterion,
      x,
      y,
      epsilon=EPSILON,
      alpha=epsilon / 4,
      step=20
  )

  with torch.no_grad():

    total += y.size(0)
    predict = clean_model(x_adv).argmax(dim=1)
    correct += (predict == y).sum().item()

accuracy = correct / total

print(f"普通训练模型的 PGD-20 准确率：{accuracy:.4f}")

普通训练模型的 PGD-20 准确率：0.0001
